# Pandas: A Friendly Survival Guide

Welcome to **Module 10 — Data Analysis with Pandas**.

NumPy is a grid of numbers. Pandas is that grid after it got column names, row labels, and a slightly judgmental attitude about missing values.

Think: Excel that went to university, then interned at an AI lab.

This notebook is a senior on the next bench, not a textbook.

> "If you select the wrong column, the model will still train. It will just learn the student ID. Then you will blame the architecture."

**How to use this:**
- Run top to bottom. Later cells reuse `df`.
- Put your own name in the table. That is when it sticks.
- Practice at the end is empty on purpose. Empty is the assignment.

**Map**
1. Intro to Pandas
2. Series and DataFrame
3. File types (CSV, JSON)
4. Basic functionalities (`head`, `info`, `describe`)
5. Build a DataFrame with columns and indexes
6. Access with column names
7. Change index / columns, `loc` and `iloc`
8. Drop, assign, iterate
9. Sort
10. Filter
11. Practice


In [ ]:
# If this cell screams, install once, then rerun.
# !pip install pandas

import pandas as pd

print("pandas:", pd.__version__)
print("Good. Spreadsheet energy is online.")


---

# 1. Intro to Pandas

You use Pandas when the data looks like a **table**:

- students and marks
- images and labels
- dates and sales

Two main objects. That is the whole religion.

| Object | In human words |
|--------|----------------|
| **Series** | one column. A list that grew an index. |
| **DataFrame** | many columns. A spreadsheet. |

If you only have a list of 4 numbers, a list is fine. If those 4 numbers have names, dates, and a missing cell, call Pandas.


In [ ]:
# A Series is one named column.
marks = pd.Series([80, 70, 90], index=["quiz1", "quiz2", "quiz3"], name="marks")
print(marks)
print("mean :", marks.mean())
print("type :", type(marks))


# 2. DataFrame and Series

A DataFrame is columns glued side by side. Each column **is** a Series.

The lazy (correct) way to build one: a dict. Keys become column names.


In [ ]:
df = pd.DataFrame({
    "name": ["Rafi", "Mim", "Nila", "Tanvir", "Joy"],
    "dept": ["CSE", "EEE", "CSE", "CSE", "EEE"],
    "age": [18, 19, 18, 20, 19],
    "marks": [80, 91, 39, 73, 66],
    "city": ["Dhaka", "Chittagong", "Dhaka", "Khulna", "Dhaka"],
})
print(df)
print()
print("one column, a Series:")
print(df["marks"])
print(type(df), type(df["marks"]))


`df["marks"]` is a Series. `df[["marks"]]` is a DataFrame with one column.

Double brackets look extra. They are extra. One is a column. Two is a tiny table. Mix them up and `.iloc[:, 0]` will haunt you.


In [ ]:
print("Series shape   :", df["marks"].shape)
print("DataFrame shape:", df[["marks"]].shape)
print(df[["name", "marks"]])


# 3. Working with different file types

Real data does not live in your homework dict. It lives in files someone emailed at 6pm.

| Type | Read | Write |
|------|------|-------|
| CSV | `pd.read_csv("file.csv")` | `df.to_csv("file.csv")` |
| JSON | `pd.read_json("file.json")` | `df.to_json("file.json")` |
| Excel | `pd.read_excel("file.xlsx")` | `df.to_excel("file.xlsx")` |

CSV is the lab default. Human readable. Excel people will still send `.xlsx`. Stay calm.

`index=False` when saving: do not write the 0,1,2,3 row numbers as a fake column. Future-you will thank present-you.


In [ ]:
df.to_csv("students.csv", index=False)
print("saved students.csv")

loaded = pd.read_csv("students.csv")
print(loaded)

df.to_json("students.json", orient="records")
print("\nfrom json:")
print(pd.read_json("students.json"))


# 4. Basic functionalities

These calls are 80% of "can you look at the data in 30 seconds?"

If you skip `info()` and jump to modeling, you will discover the age column is a string. On Friday.


In [ ]:
print("head:\n", df.head(3))     # first rows. peek, do not binge.
print("\ntail:\n", df.tail(2))   # last rows. the plot twist.
print("\nshape:", df.shape)      # (rows, columns)
print("columns:", df.columns.tolist())
print("index  :", df.index.tolist())
print("\ndtypes:\n", df.dtypes)


In [ ]:
print(df.info())
print()
print(df.describe())   # numeric columns only. "Rafi" will not get a mean.
print()
print(df["dept"].value_counts())
print("missing values:\n", df.isna().sum())


# 5. DataFrame from data, with columns and indexes

You can also build from a list of rows, then name the columns and the row index.

That index is not decoration. Later `loc` will use it like a name tag.


In [ ]:
rows = [
    [80, 70, 90],
    [60, 55, 40],
    [90, 88, 92],
]
quizzes = pd.DataFrame(
    rows,
    columns=["quiz1", "quiz2", "quiz3"],
    index=["Rafi", "Mim", "Nila"],
)
print(quizzes)
print()
print(quizzes.loc["Mim"])
print("Rafi average:", quizzes.loc["Rafi"].mean())


# 6. Accessing data with index and column

Three doors:

| Door | Meaning |
|------|---------|
| `df["marks"]` | one column by name |
| `df[["name", "marks"]]` | several columns |
| `df.loc[...]` | labels (names) |
| `df.iloc[...]` | integer positions. seats, not names. starts at 0 |

`loc` is the name. `iloc` is the seat number. Mix them up and you will invite the wrong intern to the GPU room.


In [ ]:
print(df["city"])
print()
print(df[["name", "dept", "marks"]])
print()
print("row 0 (label 0):")
print(df.loc[0])
print()
print("one cell, label style:", df.loc[2, "city"])
print("one cell, seat style :", df.iloc[2, 4])


# 7. Changing index and columns, and iloc

`set_index("name")` makes names the row labels. Then `loc["Nila"]` feels human.

`iloc` does not care about names. It cares about positions.

`rename` changes column names. Use it when the CSV column is `Marks (out of 100)  ` with a sad extra space.


In [ ]:
named = df.set_index("name")
print(named)
print()
print("Nila by name:\n", named.loc["Nila"])
print()
print("first two rows by seat:\n", named.iloc[0:2])
print()
print("first two rows, marks column by seat:\n", named.iloc[0:2, 2])


In [ ]:
renamed = df.rename(columns={"marks": "quiz_score", "city": "hometown"})
print(renamed.columns.tolist())

back = named.reset_index()
print(back.head())   # name is a column again. intern returned to the table.


Slice with `iloc[0:2]` is rows 0 and 1. Stop is excluded. Pandas inherited this pettiness from Python. Accept it.


# 8. Drop, assign values, iterate

`drop` returns a **new** DataFrame by default. The original is still there, sipping tea.

`inplace=True` edits the original. Convenient. Also how you lose a column and then gaslight yourself.

Assigning:

- new column: `df["passed"] = df["marks"] >= 40`
- one cell: `df.loc[df["name"] == "Nila", "marks"] = 45`


In [ ]:
tmp = df.copy()
print("drop city column:\n", tmp.drop(columns=["city"]))
print()
print("drop two rows:\n", tmp.drop(index=[0, 4]))
print()
print("original still has city:", "city" in tmp.columns)


In [ ]:
tmp = df.copy()
tmp["passed"] = tmp["marks"] >= 40
tmp.loc[tmp["name"] == "Nila", "marks"] = 45
tmp.loc[tmp["name"] == "Nila", "passed"] = True
print(tmp)

print("\n--- iterate rows (works, slow, honest) ---")
for i, row in tmp.iterrows():
    print(row["name"], "->", row["marks"], "| passed:", row["passed"])


`iterrows()` is for learning and for tiny tables. For 100,000 rows, use vector operations. Your CPU has a family.


# 9. Sorting a DataFrame

`sort_values("marks")` smallest to largest.

`ascending=False` is the leaderboard.

You can sort by two columns: city first, then marks. Group photo, then rank inside the group.


In [ ]:
print(df.sort_values("marks"))
print()
print(df.sort_values("marks", ascending=False))
print()
print(df.sort_values(["city", "marks"]))


# 10. Filtering based on a condition

This is boolean indexing with column names.

```text
df[df["marks"] >= 40]
```

Read it as: "from df, keep rows where that condition is True."

Several conditions: `&` for and, `|` for or. **Parentheses around each piece.** Pandas will not guess your grouping. It will scream. Let it. Then add the brackets.


In [ ]:
print("passed:\n", df[df["marks"] >= 40])
print()
print("CSE only:\n", df[df["dept"] == "CSE"])
print()
print("Dhaka AND marks >= 70:\n", df[(df["city"] == "Dhaka") & (df["marks"] >= 70)])
print()
print("CSE or EEE with isin:\n", df[df["dept"].isin(["CSE", "EEE"])])
print()
print("name is Rafi or Tanvir:\n", df[df["name"].isin(["Rafi", "Tanvir"])])


In [ ]:
# Filter, then pick columns, then sort. A tiny pipeline.
result = (
    df[df["marks"] >= 40]
    [["name", "dept", "marks"]]
    .sort_values("marks", ascending=False)
)
print(result)


---

# Pocket Recap

| Thing | One line |
|-------|----------|
| Series | one column with an index |
| DataFrame | a table. dict of columns is the easy build |
| `head` / `info` / `describe` | peek, schema, stats |
| `read_csv` / `to_csv` | disk in, disk out. `index=False` when saving |
| `df["col"]` | one Series |
| `df[["a", "b"]]` | small DataFrame |
| `loc` | labels / names |
| `iloc` | seats / positions, 0-based |
| `drop` | returns a copy unless you `inplace` |
| filter | `df[df["marks"] >= 40]` |
| `&` / `|` | and / or. Parentheses. Always. |

If you remember only three things:

1. `loc` is names. `iloc` is seats.
2. Filter with `df[condition]`, wrap each condition in `()`.
3. Save CSV with `index=False` unless you really wanted that extra column named `Unnamed: 0`.


---

# Practice — your turn

Empty cells. That is not a crash. That is the assignment.

Run the teaching cells first so `pandas` is imported and you remember the patterns.

---

## P1 — Build a table (Easy)

Make a DataFrame of 4 lab intern hours:

| name | hours | tool |
|------|-------|------|
| Rafi | 3 | numpy |
| Mim | 5 | pandas |
| Nila | 2 | numpy |
| Joy | 4 | pandas |

Print the DataFrame and its `shape`.


In [ ]:
# P1


## P2 — Peek (Easy)

Using the intern table (build it again in this cell):

- `head(2)`
- `describe()`
- `value_counts()` on `tool`


In [ ]:
# P2


## P3 — CSV round trip (Easy)

Save the intern table to `interns.csv` **without** the index. Read it back. Print the loaded table.

If the file is missing, you skipped `to_csv`. The disk is not telepathic.


In [ ]:
# P3


## P4 — loc vs iloc (Medium)

Using the intern table:

- set `name` as the index
- select Mim with `loc`
- select the first two rows with `iloc`
- print only the `hours` column for Nila using `loc`


In [ ]:
# P4


## P5 — Filter and sort (Medium)

From the intern table:

- keep rows where `hours >= 3`
- keep pandas users only
- sort by `hours` descending
- print `name` and `hours` only


In [ ]:
# P5


## P6 — New column, drop, update (Medium)

Using the intern table:

- add `overloaded` that is True when `hours >= 4`
- change Nila's hours to 3 with `loc`
- drop the `tool` column and print the result (keep the original if you want)


In [ ]:
# P6


## P7 — Two conditions (Medium)

Interns who used **numpy** **and** had **hours < 4**. Print their names.

Parentheses. `&`. You know the speech.


In [ ]:
# P7


That is Pandas.

Tables with names. `loc` for humans. `iloc` for seats. Filter before you train.

If a column looks numeric but `describe()` ignores it, check `dtypes`. If `Unnamed: 0` appears, you saved the index by accident.

See you at midterm. Bring water. Leave `inplace=True` alone unless you mean it.
